In [ ]:

@staticmethod
def read_evaluations(scenario, metrics):
    print(f"Correlations on {scenario.stem}")
    evaluation_folder = os.path.join(scenario, "evaluation")
    print('evaluation_folder', evaluation_folder)
    # Read evaluations
    evaluations = []
    for file in ['graph_evaluation.csv', 'likelihood_evaluation_USR_context300.csv',
                 f'{scenario.stem}_manual_evaluation.csv']:
        try:
            file_path = os.path.join(evaluation_folder, file)
            df = pd.read_csv(file_path, header=0, index_col='Turn')
        except:
            try:
                df = pd.read_csv(file_path, header=0, index_col='Turn', sep=';')
            except:
                print(f"Could not load {scenario}")
                df = pd.DataFrame()
                # continue

        columns_to_keep = [c for c in metrics if c in df.columns]
        df = df[columns_to_keep]
        evaluations.append(df)

    # Merge and select
    full_df = pd.concat(evaluations, axis=1)

    # rename
    # full_df.rename(columns={'System llh': 'AUTOMATIC - System llh', 'MLM llh': 'AUTOMATIC - MLM llh',
    #                         'USR DLcontext': 'AUTOMATIC - USR DLcontext', 'USR DLfact': 'AUTOMATIC - USR DLfact'},
    #                inplace=True)
    #New columns: Turn	Speaker	Cue	Response	Context	MLM response	System llh	MLM llh
    full_df.rename(columns={'System llh': 'AUTOMATIC - System llh', 'MLM llh': 'AUTOMATIC - MLM llh'},
                   inplace=True)
    full_df.rename(columns={'Overall Human Rating': 'HUMAN - Overall Human Rating',
                            'Interesting': 'HUMAN - Interesting', 'Engaging': 'HUMAN - Engaging',
                            'Specific': 'HUMAN - Specific', 'Relevant': 'HUMAN - Relevant',
                            'Correct': 'HUMAN - Correct',
                            'Semantically Appropriate': 'HUMAN - Semantically Appropriate',
                            'Understandable': 'HUMAN - Understandable',
                            'Fluent': 'HUMAN - Fluent'}, inplace=True)

    return full_df    \

In [ ]:

def correlate_metrics_scenario(self, scenario, metrics):
    # Read data from human annotations, automatic and likelihood
    convo_df = self.read_evaluations(scenario, metrics)
    # convo_df = convo_df.set_index('Turn')
    convo_df['Conversation'] = scenario.stem
    conversation_id = f"{convo_df['Conversation'].values[0]}"

    # Compute correlations
    corr_df = convo_df.corr(method='pearson', numeric_only=True)
    # Plot per scenario
    evaluation_path = os.path.join(scenario, "evaluation")
    self.plot_correlations(corr_df, None, conversation_id, evaluation_path)
    csv_file = os.path.join(evaluation_path, conversation_id+"_correlations.csv")
    corr_df.to_csv(csv_file)
    return corr_df

In [ ]:

corr_df = self.correlate_metrics_scenario(scenario, metrics)
corr_dfs.append(corr_df)

In [ ]:
import os
import pandas as pd
from emissor.persistence import ScenarioStorage
from emissor.representation.scenario import Modality
from emissor.representation.scenario import Signal, TextSignal
import emissor_util as util